# ETL Pipeline - NYC DOHMH

March 31, 2026 - Clean Data Pipeline 

## Data Overview

March 1, 2020 - Dec 31, 2021 (COVID Pandemic) ~ 12.9 million rows

| Column | Description | Data Type |
|--------|-------------|-----------|
| `extract_date` | Date of data extraction | Floating Timestamp |
| `date` | Date of emergency department visit | Floating Timestamp |
| `mod_zcta` | Modified ZIP Code tabulation area (ZCTA) of patient residence | Text |
| `total_ed_visits` | Count of all emergency department visits | Number |
| `ili_pne_visits` | Count of influenza-like illness and/or pneumonia visits | Number |
| `ili_pne_admissions` | Count of influenza-like illness and/or pneumonia visits admitted to the hospital | Number |

## Pipeline Cleaning/Transformations

- **Extract Date:** 

    - Drop column

- **Date:** 

    - Drop Missing Values
    - Ensure *Date* Data Type (mm/dd/yyyy)

- **Mod_ZCTA**

    - Drop Missing Values
    - Ensure *String* Data Type

- **Total_ED_Visits:**

    - Drop Missing Values
    - Drop Outliers (IQR Method)
    - Ensure *Integer* Data Type

- **ili_PNE_Visits:**

    - Drop Missing Values
    - Drop Outliers (IQR Method)
    - Ensure *Integer* Data Type

- **ili_PNE_Admissions:**

    - Drop Missing Values
    - Drop Outliers (IQR Method)
    - Ensure *Integer* Data Type

- **borough:**

    - feature engineered, using mod_zcta to extract boroughs

- **visit_admission_ratio**

    - ili_pne_visits / ili_pne_admissions
    - Drop Outliers (IQR Method)

In [2]:
import pandas as pd
import numpy as np

In [8]:
# Run this codeblock to clean raw CSV file
# --------------------------------------------------------------------------------

# Typical Input path: '../../Data/Emergency_Department_Visits_and_Admissions_for_Influenza-like_Illness_and_or_Pneumonia_20250714.csv'

# Typical Output path: '../../Data/nyc_dohmh_clean.csv'

# --------------------------------------------------------------------------------














path = input("Enter the path to the raw CSV file: ")
df = pd.read_csv(path)

# Output CSV to Data folder
output_path = input("Enter the path to save the cleaned CSV file: ")

# --------------------

# Cleaning pipeline for NYC DOHMH ED dataset
clean_df = df.copy()

# 1) Drop extract_date
clean_df = clean_df.drop(columns=["extract_date"], errors="ignore")

# 2) Date: drop missing + enforce datetime (mm/dd/yyyy)
clean_df["date"] = pd.to_datetime(clean_df["date"], format="%m/%d/%Y", errors="coerce")
clean_df = clean_df.dropna(subset=["date"])

# 3) Mod_ZCTA: drop missing + convert to integer ZIP
clean_df = clean_df.dropna(subset=["mod_zcta"])
clean_df["mod_zcta"] = pd.to_numeric(clean_df["mod_zcta"], errors="coerce")
clean_df = clean_df.dropna(subset=["mod_zcta"])
clean_df["mod_zcta"] = clean_df["mod_zcta"].astype(int)

# 4) Feature engineering: map Mod_ZCTA to borough and drop non-matches
zipcodes = {
    "Manhattan": list(range(10001, 10282)),
    "The Bronx": list(range(10451, 10475)),
    "Brooklyn": list(range(11201, 11256)),
    "Queens": list(range(11004, 11110)) + list(range(11351, 11698)),
    "Staten Island": list(range(10301, 10314)),
}

zip_to_borough = {z: borough for borough, zips in zipcodes.items() for z in zips}
clean_df["borough"] = clean_df["mod_zcta"].map(zip_to_borough)
clean_df = clean_df.dropna(subset=["borough"])
clean_df["mod_zcta"] = clean_df["mod_zcta"].astype(str)
clean_df["mod_zcta"] = clean_df["mod_zcta"].str.zfill(5)

# 5) Feature engineering: calculate ILI/PNE ratio, handle division by zero
clean_df["visit_admissions_ratio"] = clean_df.apply(
    lambda row: row["ili_pne_visits"] / row["total_ed_visits"]
    if row["total_ed_visits"] > 0
    else 0,
    axis=1
)

# 6) Numeric columns: drop missing, remove outliers (IQR), cast to int
numeric_cols = ["total_ed_visits", "ili_pne_visits", "ili_pne_admissions"]

for col in numeric_cols:
    clean_df[col] = pd.to_numeric(clean_df[col], errors="coerce")
    clean_df = clean_df.dropna(subset=[col])

    # IQR Method for Outliers
    q1 = clean_df[col].quantile(0.25)
    q3 = clean_df[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    clean_df = clean_df[(clean_df[col] >= lower) & (clean_df[col] <= upper)]
    clean_df[col] = clean_df[col].astype("int64")


# 7) Float column: remove outliers (IQR), keep as float
col = "visit_admissions_ratio"
q1 = clean_df[col].quantile(0.25)
q3 = clean_df[col].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
clean_df = clean_df[(clean_df[col] >= lower) & (clean_df[col] <= upper)]


# dropping rows where ratio is 0 (indicates no visits, not useful for analysis)
clean_df = clean_df[clean_df["visit_admissions_ratio"] > 0]

# Keep date format as mm/dd/yyyy in output
clean_df["date"] = clean_df["date"].dt.strftime("%m/%d/%Y")

# --------------
clean_df.to_csv(output_path, index=False)

print(f"Saved cleaned file to: {output_path}")
print(f"Final shape: {clean_df.shape}")

/var/folders/gs/yv3dj4zx2tgdd5tw701ypspm0000gp/T/ipykernel_11113/1153871408.py:24: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


Saved cleaned file to: ../../Data/nyc_dohmh_clean.csv
Final shape: (7454806, 7)


In [9]:
clean_df.head()

,date,mod_zcta,total_ed_visits,ili_pne_visits,ili_pne_admissions,borough,visit_admissions_ratio
0,03/01/2020,11692,16,2,0,Queens,0.125000
3,03/01/2020,11224,72,2,1,Brooklyn,0.027778
4,03/01/2020,11419,52,5,0,Queens,0.096154
5,03/01/2020,10028,19,1,0,Manhattan,0.052632
7,03/01/2020,10307,16,1,1,Staten Island,0.062500
